# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*

## 👥 Team Information
- **Team #:** ____
- **Members:** Name 1, Name 2, Name 3


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Manual IR and Inverted Matrix coding + Markdown documentation (work as teams)
3. **Push to GitHub** *(15 min)* – Teams commit and push initial notebooks. **Make sure to include your names so it is easy to identify the team that developed the Min-Max code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(15 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. **Push to the course GitHub** *(automated)* – Find the last code cell in this notebook, update your team number, and run the cell. It will push your notebook to the repo at `submissions/team#`

## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and Inverted Index.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [1]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        # Clean filename
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'data', max_posts=20)

Saved 20 blog posts to data


In [2]:
# Example: Load text files from a folder
import os

def load_documents(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

# Replace 'data/' with your actual folder
documents = load_documents('data/')
print(f"Loaded {len(documents)} documents.")


Loaded 20 documents.


### 🗣 Team Talking Point (Step 1):
> `os.listdir()` does not return files in a fixed order. When we ran the notebook twice, "Document 0" was a different blog post each time, so our doc IDs and results changed between runs. We sort the file names so every run gives the same doc IDs. We also check that our vocabulary is over 2000 unique words, as the task asks.

In [3]:
import re

def load_documents_sorted(folder_path):
    documents = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

documents = load_documents_sorted('data/')
print(f"Loaded {len(documents)} documents.")

# Check the size of the vocabulary
vocabulary = set()
for doc in documents:
    vocabulary.update(re.findall(r'\b\w+\b', doc.lower()))
print(f"Unique words in the collection: {len(vocabulary)}")

Loaded 20 documents.
Unique words in the collection: 2725


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [4]:
import re

def tokenize(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])  # Preview first 20 tokens


['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'p', 'b', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to']


### 🗣 Team Talking Point (Step 2):
> The RSS feed gives us the posts as **HTML**, not plain text. Because of that, HTML tags became tokens, for example `'p'` from `<p>`, and words like `href` or `https` from links. These are not real words from the post. We fixed the tokenizer so it removes HTML tags first and then splits the text into words.

In [5]:
import html

def clean_html(text):
    text = re.sub(r'<[^>]+>', ' ', text)  # remove tags like <p> or <a href="...">
    text = html.unescape(text)            # turn codes like &amp; into &
    return text

def tokenize(text):
    text = clean_html(text)
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

print("Old tokens:", tokens[:20])
tokens = tokenize(documents[0])
print("New tokens:", tokens[:20])

Old tokens: ['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'p', 'b', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to']
New tokens: ['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to', 'announce', 'that']


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [6]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def normalize_tokens(tokens):
    return [stemmer.stem(t) for t in tokens if t not in stop_words]

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])


['reopen', 'psf', 'grant', 'program', 'twice', 'yearli', 'fund', 'round', 'python', 'softwar', 'foundat', 'psf', 'excit', 'announc', 'psf', 'grant', 'program', 'reopen', 'restructur', 'schedul']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Davisen\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 🗣 Team Talking Point (Step 3):
> The Porter stemmer is fast but not perfect. It creates words that are not real (`notice` → `notic`), it can join different words (`university` and `universe` both become `univers`), and it misses irregular forms (`ran` stays `ran`). This means the **query must be normalized the same way as the documents**, or it will not match anything. Stop-word removal also makes the documents much shorter.

In [7]:
for word in ['notice', 'running', 'runs', 'ran', 'university', 'universe']:
    print(word, '->', stemmer.stem(word))

norm_tokens = normalize_tokens(tokens)
print("Tokens before normalization:", len(tokens))
print("Tokens after normalization: ", len(norm_tokens))

notice -> notic
running -> run
runs -> run
ran -> ran
university -> univers
universe -> univers
Tokens before normalization: 1033
Tokens after normalization:  627


## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [8]:
from collections import defaultdict

def build_inverted_index(documents):
    index = defaultdict(list)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        seen = set()
        for token in tokens:
            if token not in seen:
                index[token].append(doc_id)
                seen.add(token)
    return dict(index)

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))  # Preview first 10 terms


{'reopen': [0], 'psf': [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19], 'grant': [0, 5, 8, 11, 12, 13, 15, 16, 18, 19], 'program': [0, 2, 3, 5, 6, 8, 10, 11, 12, 13, 15, 16, 17, 18], 'twice': [0], 'yearli': [0, 2], 'fund': [0, 5, 10, 11, 12, 13, 14, 18], 'round': [0, 5, 13, 18], 'python': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'softwar': [0, 1, 3, 5, 6, 8, 11, 12, 13, 14, 15, 16, 17, 19]}


### 🗣 Team Talking Point (Step 4):
> The basic inverted index only tells us **which** documents contain a word, not **where** the word is. So we can do a Boolean AND query (both words appear somewhere in the document), but that is not the same as a phrase query (the words appear next to each other). To support phrases, we build a **positional index** that also saves the position of each word.

In [9]:
def and_query(query, index):
    terms = normalize_tokens(tokenize(query))
    if not terms:
        return []
    result = set(index.get(terms[0], []))
    for term in terms[1:]:
        result = result & set(index.get(term, []))
    return sorted(result)

def build_pos_index(documents):
    pos_index = {}
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            pos_index.setdefault(token, {}).setdefault(doc_id, []).append(pos)
    return pos_index

pos_index = build_pos_index(documents)

print("AND query 'python' + 'board':", and_query("python board", inverted_index))
print("Positions of 'python' in document 0:", pos_index.get('python', {}).get(0, []))

AND query 'python' + 'board': [0, 3, 4, 5, 6, 8, 9, 10, 12, 13, 17, 18]
Positions of 'python' in document 0: [8, 50, 195, 413, 449, 543, 575, 578, 589, 598, 602, 615, 625]


## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries.
- Demonstrate that they return the correct documents.


In [10]:
# Placeholder for phrase query implementation
# You may build a position-aware index or use string search within docs after normalization

# Example:
query1 = "Python software"
query2 = "bylaws"

def phrase_query(query, documents):
    results = []
    query_tokens = normalize_tokens(tokenize(query))
    for doc_id, doc in enumerate(documents):
        doc_tokens = normalize_tokens(tokenize(doc))
        # Search for the sequence in the token list
        for i in range(len(doc_tokens) - len(query_tokens) + 1):
            if doc_tokens[i:i+len(query_tokens)] == query_tokens:
                results.append(doc_id)
                break
    return results

# Run phrase queries on the loaded documents
results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

Documents containing the phrase 'Python software': [0, 1, 3, 5, 6, 11, 13, 14, 15, 16, 17, 19]
Documents containing the phrase 'bylaws': [3, 6]


### 🗣 Team Talking Point (Phrase Queries):
> The phrase query above normalizes and scans **every document** for every query, which gets slow with a big collection. With the positional index we only look at documents that contain the first word, and then check if the next words are at position +1, +2, and so on. Note: stop words are removed, so "board **of** directors" is searched as "board director".

In [11]:
def phrase_query_positional(query, pos_index):
    terms = normalize_tokens(tokenize(query))
    if not terms:
        return []
    results = []
    for doc_id, positions in pos_index.get(terms[0], {}).items():
        for p in positions:
            match = True
            for i in range(1, len(terms)):
                if p + i not in pos_index.get(terms[i], {}).get(doc_id, []):
                    match = False
                    break
            if match:
                results.append(doc_id)
                break
    return sorted(results)

# Our 2 phrase query tests
our_query1 = "Python Software Foundation"
our_query2 = "board of directors"

for q in [our_query1, our_query2]:
    print(f"Phrase query '{q}' -> searched as {normalize_tokens(tokenize(q))}")
    print("  Positional index result:", phrase_query_positional(q, pos_index))
    print("  Scanning result (check):", phrase_query(q, documents))
    print("  AND query (no order):   ", and_query(q, inverted_index))

Phrase query 'Python Software Foundation' -> searched as ['python', 'softwar', 'foundat']
  Positional index result: [0, 1, 3, 5, 6, 11, 13, 14, 15, 16, 17, 19]
  Scanning result (check): [0, 1, 3, 5, 6, 11, 13, 14, 15, 16, 17, 19]
  AND query (no order):    [0, 1, 3, 5, 6, 11, 13, 14, 15, 16, 17, 19]
Phrase query 'board of directors' -> searched as ['board', 'director']
  Positional index result: [3, 5, 6, 8, 9, 10, 12, 17, 18]
  Scanning result (check): [3, 5, 6, 8, 9, 10, 12, 17, 18]
  AND query (no order):    [3, 5, 6, 8, 9, 10, 12, 17, 18]


## 🧠 Additional Challenge: Implement Positional Indexes

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [12]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

Positions for term 'python':
  Document 0: [8, 50, 195, 413, 449, 543, 575, 578, 589, 598, 602, 615, 625]
  Document 1: [1, 28, 54, 60, 65, 68, 74, 78, 89, 123, 129, 151]
  Document 2: [0, 11, 17, 30, 45, 60, 95, 109, 143, 174]
  Document 3: [17, 175, 216, 250, 279, 333]
  Document 4: [2, 8, 45, 58, 210]
  Document 5: [6, 68, 243]
  Document 6: [57, 108, 212]
  Document 7: [1, 13, 49, 74, 102, 115, 176]
  Document 8: [54, 103, 166, 199, 234, 269, 273, 296]
  Document 9: [47, 63, 70, 169, 175, 185, 196, 209, 223, 251, 266, 290, 308, 315, 325, 335, 373, 383, 399, 411, 438, 468]
  Document 10: [8, 16, 20, 47, 181, 194, 223, 236, 326]
  Document 11: [82, 90, 453, 464, 475, 484, 488, 501]
  Document 12: [19, 26, 33, 49, 80, 158, 183, 219, 257, 327, 392, 435]
  Document 13: [0, 9, 16, 21, 43, 56, 67, 87, 93, 101, 112, 182, 190, 203, 210, 230, 237, 258, 266, 275, 279, 328, 343, 345, 364, 373, 389, 401, 507, 564, 578, 627, 641, 708, 767, 842]
  Document 14: [0, 32, 45, 48, 54, 59, 105, 124, 14

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

### 📝 Completed Table: Term Frequency (TF) and Inverse Document Frequency (IDF weight)

| Term                           | What is it?                                                                 | How is it used?                          | Pros                                   | Cons                                   |
|-------------------------------|----------------------------------------------------------------------------|------------------------------------------|----------------------------------------|----------------------------------------|
| Term Frequency (TF)            | Number of times a term appears in a document                               | Measures term importance in a document   | Simple, fast, good for ranking         | Doesn't account for common terms       |
| Inverse Document Frequency     | Log ratio of total docs to docs containing the term                        | Weights rare terms higher                | Reduces impact of common words         | Can overweight rare, irrelevant terms  |
| (IDF weight)                   | Used with TF to score term relevance: TF-IDF = TF * IDF                    | Improves search relevance                | Highlights unique, important terms     | Requires global corpus statistics      |

### 🗣 Team Talking Points: Searching bigrams (biwords)
- **We prefer the positional index for bigrams.** For a bigram like "machine learning", we find the positions of the first word and check if the second word is at the next position (+1). This gives exact matches.
- A **Term Document Count Matrix** only stores how many times each word appears in each document (the TF). It does not know the order of words, so it can only tell us that both words appear somewhere in the document. This can give wrong matches.
- **TF and IDF are still useful**: after the positional index finds the documents that contain the bigram, we can use TF-IDF to rank them, so documents where the bigram is frequent and rare in the collection come first.
- Downside: the positional index uses more memory because it stores every position.

Let's implement the Inverse Document Frequency (IDF)

In [13]:
# Calculate Inverse Document Frequency (IDF) for each term using the positional index
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}
    for term, doc_positions in positional_index.items():
        doc_count = len(doc_positions)
        if doc_count > 0:
            # --- IGNORE ---
            # print(f"Computing IDF for {term} '{total_docs}' / {doc_count}")
            idf_scores[term] = math.log(total_docs / doc_count)
        else:
            idf_scores[term] = 0.0
    return idf_scores

# Compute IDF for all terms using the positional index
idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# Example: Show IDF for a sample term
# NOTE : We applied stemming in the normalization step, so the term may not match exactly if it was stemmed
sample_term = 'softwar'  # Change to any term you want to inspect
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

IDF for term 'softwar': 0.3566749439387324


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

In [14]:
# Calculate Term Frequency (TF) for the term 'softwar' in each document
# TF is simply the number of times the term appears in a document
import pandas as pd

term = 'softwar'  # Use the stemmed term as before

tf_values = []
for doc_id, doc in enumerate(documents):
    tokens = normalize_tokens(tokenize(doc))
    tf = tokens.count(term)
    tf_values.append(tf)
    print(f"TF for term '{term}' in Document {doc_id}: {tf}")

# Display as a DataFrame for clarity
TF_df = pd.DataFrame({'Document': list(range(len(documents))), 'TF': tf_values})
print(TF_df)

TF for term 'softwar' in Document 0: 3
TF for term 'softwar' in Document 1: 1
TF for term 'softwar' in Document 2: 0
TF for term 'softwar' in Document 3: 1
TF for term 'softwar' in Document 4: 0
TF for term 'softwar' in Document 5: 1
TF for term 'softwar' in Document 6: 1
TF for term 'softwar' in Document 7: 0
TF for term 'softwar' in Document 8: 1
TF for term 'softwar' in Document 9: 0
TF for term 'softwar' in Document 10: 0
TF for term 'softwar' in Document 11: 2
TF for term 'softwar' in Document 12: 2
TF for term 'softwar' in Document 13: 1
TF for term 'softwar' in Document 14: 6
TF for term 'softwar' in Document 15: 3
TF for term 'softwar' in Document 16: 6
TF for term 'softwar' in Document 17: 3
TF for term 'softwar' in Document 18: 0
TF for term 'softwar' in Document 19: 1
    Document  TF
0          0   3
1          1   1
2          2   0
3          3   1
4          4   0
5          5   1
6          6   1
7          7   0
8          8   1
9          9   0
10        10   0
11    

#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.

### 🗣 Team Talking Point (Query Processor):
> The original `query_processor` used `query.split()` directly, so the query was not lowercased or stemmed. A search for "Software" would never match the index term `softwar`. We pass the query through the same `tokenize` and `normalize_tokens` steps as the documents. We also added a simple ranked search that sorts documents by their TF-IDF score.

In [15]:
# Combine TF and IDF to compute TF-IDF for the term 'softwar' in each document
tfidf_values = []
for doc_id in range(len(documents)):
    tfidf_values.append(tf_values[doc_id] * idf_scores.get(term, 0.0))

TF_df['TF-IDF'] = tfidf_values
print(TF_df)

# Query processor: the query must be tokenized and normalized like the documents
def query_processor(query, inverted_index):
    results = []
    for term in normalize_tokens(tokenize(query)):
        if term in inverted_index:
            results.extend(inverted_index[term])
    return sorted(set(results))

# Ranked search: add up TF-IDF of each query word in each document
def ranked_search(query, positional_index, idf_scores):
    scores = {}
    for term in normalize_tokens(tokenize(query)):
        for doc_id, positions in positional_index.get(term, {}).items():
            tf = len(positions)
            scores[doc_id] = scores.get(doc_id, 0) + tf * idf_scores.get(term, 0.0)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)

print("Documents for 'Software bylaws':", query_processor("Software bylaws", inverted_index))
print("Top 5 ranked documents for 'Software bylaws':")
for doc_id, score in ranked_search("Software bylaws", positional_index, idf_scores)[:5]:
    print(f"  Document {doc_id}: {score:.3f}")

    Document  TF    TF-IDF
0          0   3  1.070025
1          1   1  0.356675
2          2   0  0.000000
3          3   1  0.356675
4          4   0  0.000000
5          5   1  0.356675
6          6   1  0.356675
7          7   0  0.000000
8          8   1  0.356675
9          9   0  0.000000
10        10   0  0.000000
11        11   2  0.713350
12        12   2  0.713350
13        13   1  0.356675
14        14   6  2.140050
15        15   3  1.070025
16        16   6  2.140050
17        17   3  1.070025
18        18   0  0.000000
19        19   1  0.356675
Documents for 'Software bylaws': [0, 1, 3, 5, 6, 8, 11, 12, 13, 14, 15, 16, 17, 19]
Top 5 ranked documents for 'Software bylaws':
  Document 6: 4.962
  Document 3: 2.659
  Document 14: 2.140
  Document 16: 2.140
  Document 0: 1.070


## 🧠 Reflection
- Small details in each step change the final results: file order changed our doc IDs, and HTML tags added fake words to our vocabulary.
- The query must go through the same tokenizer and normalization as the documents, otherwise words like "Software" never match `softwar`.
- A normal inverted index is good for "which documents contain this word", but phrase queries need a positional index.
- Removing stop words and stemming makes the index smaller, but it can also change the meaning of a phrase (e.g. "board of directors" becomes "board director").
- TF-IDF helps us rank results, so the most relevant documents come first.

## 🧠 Push your code to the course GitHub repository

Update your team number and then run the code. 

In [16]:
#!python submit_assignment.py --notebook IR_InvertedMatrix_Workshop.ipynb --student_id "teamX"  # change teamX to your team number